In [1]:
import multiprocessing
import time
import random

Imagine que sua matriz tem 300 linhas e você tem 4 núcleos (CPUs) no seu computador. A função multiplicar_paralelo divide essas 300 linhas em 4 blocos de 75 linhas cada.

O Processo 1 recebe as linhas 0 a 74.

O Processo 2 recebe as linhas 75 a 149, e assim por diante.

A função multiplicar_parcial é justamente o que cada um desses processos executa de forma isolada e simultânea.

Ela recebe um pacote de dados (args) que contém:

As matrizes completas A e B.

O índice de inicio e fim das linhas que aquele processo específico deve calcular.

Independência: Cada chamada de multiplicar_parcial não sabe que as outras existem. Elas não precisam esperar umas pelas outras.

Retorno: No final, ela retorna apenas o pedaço (a "fatia") da matriz que ela calculou.

Reunião: Depois que todas as funções multiplicar_parcial terminam, a função principal junta todos os resultados (C.extend(parte)) para formar a matriz final.

In [2]:
def multiplicar_parcial(args):
    A, B, inicio, fim = args
    colunas_b = len(B[0])
    linhas_b = len(B)
    resultado_parcial = [[0 for _ in range(colunas_b)] for _ in range(fim - inicio)] # Inicializa a matriz de resultado parcial
    for i in range(inicio, fim):
        for j in range(colunas_b):
            for k in range(linhas_b):
                resultado_parcial[i - inicio][j] += A[i][k] * B[k][j]
    return resultado_parcial


Cria uma matriz cheia de zeros logo de cara: [[0 for _ in range(colunas_b)] for _ in range(fim - inicio)]. Depois, ela apenas substitui o valor 0 pelo resultado final.

olha especificamente para o número de colunas de B (len(B[0])) e o número de linhas de B (len(B)), o que é o correto para a regra de multiplicação de matrizes. Ela lida corretamente com matrizes que não possuem o mesmo número de linhas e colunas.

In [3]:
def multiplicar_paralelo(A, B, num_processos):
    linhas_a = len(A)
    colunas_b = len(B[0])
    resultado = [[0 for _ in range(colunas_b)] for _ in range(linhas_a)] # Inicializa a matriz de resultado
    pool = multiprocessing.Pool(processes=num_processos) #reserva uma quantidade específica de núcleos da sua CPU para ficarem dedicados ao seu programa.
    tarefas = []
    linhas_por_processo = (linhas_a + num_processos - 1) // num_processos # Calcula o número de linhas por processo
    for i in range(num_processos):
        inicio = i * linhas_por_processo
        fim = min(inicio + linhas_por_processo, linhas_a)
        if inicio < fim: # Verifica se há linhas para processar
            tarefas.append((A, B, inicio, fim))
    resultados_parciais = pool.map(multiplicar_parcial, tarefas) # Executa as tarefas em paralelo
    pool.close() # Fecha o pool para não aceitar mais tarefas
    pool.join() # Aguarda a conclusão de todas as tarefas (sincroniza os processos)
    # Combina os resultados parciais na matriz de resultado final
    for i, resultado_parcial in enumerate(resultados_parciais):
        inicio = i * linhas_por_processo
        for j in range(len(resultado_parcial)):
            resultado[inicio + j] = resultado_parcial[j]
    return resultado


In [4]:
def gerar_matriz(linhas, colunas):
    return [[random.randint(1, 10) for _ in range(colunas)] for _ in range(linhas)]

In [5]:
n = 1000
m = 2000
A = gerar_matriz(n, m)
B = gerar_matriz(m, n)


In [ ]:
inicio = time.time()
C = multiplicar_paralelo(A, B, 4)
fim = time.time()
print(f"Tempo de execução: {fim - inicio:.2f} segundos")

n=10
m=20
Tempo de execução: 0.19 segundos

n=100
m=200
Tempo de execução: 0.24 segundos

n=1000
m=2000
Tempo de execução: 60.53 segundos